# Bottleneck Deconv Training - All Latent Dimensions
Training ComplexBottleneckDeconvDecoder with BottleneckEncoder across multiple latent dimensions

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import time
from tqdm.auto import tqdm

# Runtime reload
import importlib
import models.autoencoder.bottleneck_models
importlib.reload(models.autoencoder.bottleneck_models)

from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.autoencoder.bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckAE,
)
from models.autoencoder.training import train_autoencoder
from models.autoencoder.config import TrainingConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Data Setup

In [ ]:
root_dir = Path(".")
images_dir = root_dir / "data" / "Images"
mask_path = root_dir / "data" / "masks" / "rmask_ICV.nii"

# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 4
output_dir = "output/Experiments/BottleneckDeconvTraining"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)
display(df.head())

# Create dataloaders
print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    df, batch_size=batch_size, train_split=0.8, 
    on_demand=True, mask_path=mask_path,
    num_workers=2
)

print(f"Data preparation complete. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

## Model Setup

In [ ]:
target_shape = (64, 128, 128)

def build_model(latent_dim=128):
    """Build ComplexBottleneckDeconvDecoder model"""
    model = BottleneckAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
    )
    return model.to(device)

def create_training_config(model_name, epochs=10):
    """Create a training configuration for a model"""
    config = TrainingConfig()
    config.epochs = epochs
    config.model_name = model_name
    config.checkpoint_dir = os.path.join(output_dir, model_name)
    config.early_stopping_patience = 5
    config.save_interval = 5
    config.learning_rate = 1e-4
    config.use_mixed_precision = True
    return config

## Training All Latent Dimensions

In [ ]:
# Latent dimensions to test
deconv_latent_dims = [512, 256, 128, 64, 32]
deconv_histories = {}
deconv_results = []

num_epochs = 200

for idx, latent_dim in enumerate(deconv_latent_dims):
    model_name = f"deconv_lat{latent_dim}"
    print(f"\n[{idx+1}/{len(deconv_latent_dims)}] ================== Training {model_name} ==================")
    
    cfg = create_training_config(model_name, num_epochs)
    model = build_model(latent_dim=latent_dim)
    
    # Count parameters
    num_params = count_trainable(model)
    print(f"Parameters: {num_params / 1e6:.2f}M")
    
    # Time the training
    start_time = time.time()
    
    train_losses, val_losses, trained_model = train_autoencoder(
        model,
        train_loader,
        val_loader,
        config=cfg,
        disable_epoch_bars=True,
        criterion=None,  # default MSE
    )
    
    elapsed_time = time.time() - start_time
    time_per_epoch = elapsed_time / num_epochs
    
    deconv_histories[model_name] = {"train": train_losses, "val": val_losses}
    deconv_results.append({
        "Model": model_name,
        "Latent Dim": latent_dim,
        "Parameters": num_params,
        "Final Val Loss": val_losses[-1],
        "Best Val Loss": min(val_losses),
        "Final Train Loss": train_losses[-1],
        "Total Time (s)": elapsed_time,
        "Time per Epoch (s)": time_per_epoch,
    })
    
    print(f"✓ Completed in {elapsed_time:.1f}s | Time/Epoch: {time_per_epoch:.2f}s | Final Val Loss: {val_losses[-1]:.6f}")

## Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(deconv_results)
display(results_df)

# Save results
results_df.to_csv(os.path.join(output_dir, "training_results.csv"), index=False)
print(f"\nResults saved to {os.path.join(output_dir, 'training_results.csv')}")

## Visualization

In [ ]:
# Plot training curves for all models
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, history) in enumerate(deconv_histories.items()):
    ax = axes[idx]
    ax.plot(history['train'], label='Train Loss', linewidth=2)
    ax.plot(history['val'], label='Val Loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(model_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "training_curves.png"), dpi=150)
plt.show()

print(f"Training curves saved to {os.path.join(output_dir, 'training_curves.png')}")

In [ ]:
# Compare final losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Final validation loss vs latent dim
latent_dims = results_df['Latent Dim'].values
final_val_losses = results_df['Final Val Loss'].values
best_val_losses = results_df['Best Val Loss'].values

axes[0].plot(latent_dims, final_val_losses, 'o-', label='Final Val Loss', linewidth=2, markersize=8)
axes[0].plot(latent_dims, best_val_losses, 's--', label='Best Val Loss', linewidth=2, markersize=8)
axes[0].set_xlabel('Latent Dimension')
axes[0].set_ylabel('Loss')
axes[0].set_title('Validation Loss vs Latent Dimension')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].invert_xaxis()

# Parameters vs latent dim
params = results_df['Parameters'].values / 1e6
axes[1].plot(latent_dims, params, 'o-', color='green', linewidth=2, markersize=8)
axes[1].set_xlabel('Latent Dimension')
axes[1].set_ylabel('Parameters (M)')
axes[1].set_title('Model Parameters vs Latent Dimension')
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis()

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "comparison_analysis.png"), dpi=150)
plt.show()

print(f"Comparison analysis saved to {os.path.join(output_dir, 'comparison_analysis.png')}")